# 레슨 09 - 메타인지 디자인 패턴


## 설정

이 노트북은 Microsoft Agent Framework를 사용하여 메타인지 설계 패턴을 시연합니다.

**사전 요구 사항:**
- 환경 변수를 통해 구성된 Azure OpenAI 배포
- Azure CLI 인증 (`az login`)


In [ ]:
! pip install agent-framework azure-ai-projects azure-identity -q

In [ ]:
import logging
logging.getLogger("agent_framework.azure").setLevel(logging.ERROR)

import os
import asyncio
from typing import Annotated

from agent_framework import tool
from agent_framework.azure import AzureAIProjectAgentProvider
from azure.identity import AzureCliCredential

In [ ]:

provider = AzureAIProjectAgentProvider(credential=AzureCliCredential())

## 메타인지란 무엇인가?

메타인지는 **생각에 대한 사고**이다. AI 에이전트의 맥락에서, 이는 다음을 수행할 수 있는 에이전트를 구축하는 것을 의미한다:

- **자기 성찰**하여 자신의 출력과 추론 과정을 검토한다
- **오류를 감지**하고 조용히 실패하는 대신 우아하게 복구한다
- **평가**하여 응답이 완전하고 도움이 되는지 판단한다
- **적응**하여 초기 접근 방식이 작동하지 않을 때 전략을 바꾼다 (예: 백업 시스템으로 되돌아가기)

메타인지적 에이전트는 단순히 질문에 답하는 것에 그치지 않는다 — 스스로의 성능을 모니터링하고 실시간으로 조정한다.


## 기본 및 백업 도구

일반적인 메타인지 패턴은 **대체 전략**입니다. 에이전트는 먼저 기본 도구를 시도합니다; 실패할 경우(예: 404 오류), 에이전트는 실패를 인식하고 투명하게 백업 도구로 전환합니다.

이는 주요 서비스가 사용 불가일 수 있는 실제 시스템을 반영하며, 에이전트는 대안 경로를 선택하기 전에 스스로 문제를 진단해야 합니다.

아래에서는 두 가지 항공편 조회 도구를 정의합니다:
- **기본** — 파리, 도쿄, 바르셀로나를 포함합니다
- **백업** — 베를린, 시드니, 뉴욕을 포함합니다


In [ ]:
@tool(approval_mode="never_require")
def get_flight_times(
    destination: Annotated[str, "The destination city"]
) -> str:
    """Get available flight times for a destination (primary source)."""
    flights = {
        "Paris": "Departures: 08:00, 12:30, 17:45 — from $350",
        "Tokyo": "Departures: 11:00, 23:30 — from $890",
        "Barcelona": "Departures: 07:15, 14:00, 19:30 — from $280",
    }
    if destination in flights:
        return flights[destination]
    raise Exception(f"404: No flights found for {destination} in primary system")


@tool(approval_mode="never_require")
def get_flight_times_backup(
    destination: Annotated[str, "The destination city"]
) -> str:
    """Get available flight times from backup system (used when primary fails)."""
    backup_flights = {
        "Berlin": "Departures: 09:00, 16:00 — from $220",
        "Sydney": "Departures: 22:00 — from $1200",
        "New York City": "Departures: 06:00, 10:30, 15:00, 20:00 — from $450",
    }
    return backup_flights.get(
        destination,
        f"No flights found for {destination} in any system. Please try again later.",
    )

## 오류 복구 기능이 있는 자기 성찰 에이전트

아래의 에이전트는 우선 기본 비행 시스템을 먼저 시도하고, 실패를 인식하며, 투명하게 백업 시스템으로 전환하도록 지시받았습니다. 각 응답 후에는 사용자의 질문에 완전히 답변했는지 간단히 자기평가합니다.


In [ ]:
agent = await provider.create_agent(
    tools=[get_flight_times, get_flight_times_backup],
    name="FlightBookingAgent",
    instructions="""You are a flight booking agent with self-reflection capabilities.

When looking up flights:
1. Try the primary flight system first (get_flight_times)
2. If the primary system fails (404 error), acknowledge the error and try the backup system (get_flight_times_backup)
3. Always explain to the user what happened — be transparent about fallbacks
4. If both systems fail, apologize and suggest alternatives

After each response, briefly evaluate whether your answer was complete and helpful.""",
)

# Test with a destination in primary system
print("=== Test 1: Destination in primary system ===")
response = await agent.run(
    "What flights are available to Paris?",
    )
print(response)

# Test with a destination only in backup system
print("\n=== Test 2: Destination only in backup system ===")
response = await agent.run(
    "What flights are available to Berlin?",
    )
print(response)

## 자기 평가 패턴

메타인지의 또 다른 측면은 **자기 평가**입니다: 별도의 에이전트(또는 동일한 에이전트가 두 번째 검토에서)가 응답의 완전성, 정확성 및 유용성을 검토합니다.

아래에서는 여행 에이전트의 응답을 세 가지 차원에서 점수화하는 `ResponseEvaluator` 에이전트를 만듭니다.


In [ ]:
evaluation_agent = await provider.create_agent(
    tools=[get_flight_times, get_flight_times_backup],
    name="ResponseEvaluator",
    instructions="""You are a quality evaluator for travel agent responses.
Given a travel question and the agent's response, evaluate:
1. Completeness: Did it answer all parts of the question? (1-5)
2. Accuracy: Is the information correct? (1-5)
3. Helpfulness: Would a traveler find this useful? (1-5)
Provide a brief evaluation with scores and one suggestion for improvement.""",
)

# Evaluate the agent's response from Test 1
eval_prompt = f"""Question: What flights are available to Paris?
Agent Response: {response}

Please evaluate the above response."""

evaluation = await evaluation_agent.run(eval_prompt)
print("=== Self-Evaluation ===")
print(evaluation)

## 요약

이번 레슨에서는 Microsoft Agent Framework를 사용하여 **메타인지 에이전트**를 구축하는 방법을 배웠습니다:

- **자기성찰**: 자신의 추론을 모니터링하고 무슨 일이 있었는지 투명하게 전달하는 에이전트들.
- **폴백을 통한 오류 복구**: 에이전트가 실패(예: 404 오류)를 감지하면 자동으로 대체 소스를 시도하는 주 도구 + 백업 도구 패턴.
- **자기평가**: 응답의 완전성, 정확성, 유용성을 점수화하는 별도의 평가 에이전트.

이러한 패턴은 에이전트를 더 견고하고 투명하며 신뢰할 수 있게 만들어 줍니다 — 운영 환경에 배포할 때 중요한 특성입니다.


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**면책사항**:
이 문서는 AI 번역 서비스 [Co-op Translator](https://github.com/Azure/co-op-translator)를 사용하여 번역되었습니다. 정확성을 위해 노력하고 있지만 자동 번역에는 오류나 부정확성이 포함될 수 있음을 유의하시기 바랍니다. 원본 문서를 권위 있는 출처로 간주해야 합니다. 중요한 정보의 경우 전문적인 인간 번역을 권장합니다. 본 번역의 사용으로 인해 발생하는 오해나 잘못된 해석에 대해서는 책임을 지지 않습니다.
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
